# 5. Kreiranje dimenzijskog modela (Star shema)

Skripta za generiranje dimenzijskog modela podataka → star schema.

Dimenzijski model podataka je zvjezdasti model koji se sastoji od jedne tablice činjenica
i više tablica dimenzija (data mart). Ovom skriptom samo stvaramo shemu,
popunjavanje ostavljamo za ETL proces (notebook 6).

### Dimenzije:
- `dim_tehnicar` — tehničari (reporter i assignee)
- `dim_projekt_prioritet_status` — junk dimenzija: projekt × prioritet × status
- `dim_vrijeme` — vremenska dimenzija

### Fact tablica:
- `fact_support_tickets` — sadrži metrike i strane ključeve prema dimenzijama

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, Column, Integer, BigInteger, String, Date, Float, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

load_dotenv()

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'fipu_srp_projekt')

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD nije postavljen! Kreiraj .env datoteku.")

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
engine = create_engine(DATABASE_URL, echo=False)
Session = sessionmaker(bind=engine)
session = Session()
Base = declarative_base()

print(f"Spojeno na bazu: {DB_NAME} ({DB_HOST})")

Spojeno na bazu: fipu_srp_projekt (localhost)


C:\Users\ipavl\AppData\Local\Temp\ipykernel_26208\2259122783.py:21: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


## 5.1 Definicija dimenzijskih tablica (ORM klase)

In [2]:
class DimTehnicar(Base):
    """Dimenzija tehničara — reporteri i assigneei (100 osoba)."""
    __tablename__ = 'dim_tehnicar'

    tehnicar_key = Column(Integer, primary_key=True, autoincrement=True)
    ime_prezime = Column(String(255))


class DimProjektPrioritetStatus(Base):
    """
    Junk dimenzija — kombinacija projekt × prioritet × status.
    Spaja tri kategorijske varijable u jednu dimenziju.
    Teoretski max: 15 × 7 × 15 = 1.575 kombinacija.
    Stvarni broj: ~330 (samo postojeće kombinacije iz podataka).
    """
    __tablename__ = 'dim_projekt_prioritet_status'

    projekt_prioritet_status_key = Column(Integer, primary_key=True, autoincrement=True)
    naziv_projekta = Column(String(255))
    razina_prioriteta = Column(String(50))
    naziv_statusa = Column(String(50))


class DimTipTicketa(Base):
    """Dimenzija tipa ticketa — 15 tipova (Ticket, Service, Story, Task...)."""
    __tablename__ = 'dim_tip_ticketa'

    tip_ticketa_key = Column(Integer, primary_key=True, autoincrement=True)
    naziv_tipa = Column(String(50))


class DimRezolucija(Base):
    """Dimenzija rezolucije — 4 tipa (Done, Won't Do, Duplicate, Cannot Reproduce)."""
    __tablename__ = 'dim_rezolucija'

    rezolucija_key = Column(Integer, primary_key=True, autoincrement=True)
    naziv_rezolucije = Column(String(50))
    naziv_statusa = Column(String(50))


class DimVrijeme(Base):
    """Vremenska dimenzija — jedan redak po datumu kreiranja ticketa."""
    __tablename__ = 'dim_vrijeme'

    vrijeme_key = Column(Date, primary_key=True)
    dan = Column(Integer)
    mjesec = Column(Integer)
    godina = Column(Integer)
    kvartal = Column(Integer)
    dan_u_tjednu = Column(String(20))


print("Dimenzijske klase definirane: DimTehnicar, DimProjektPrioritetStatus, DimTipTicketa, DimRezolucija, DimVrijeme")

Dimenzijske klase definirane: DimTehnicar, DimProjektPrioritetStatus, DimTipTicketa, DimRezolucija, DimVrijeme


## 5.2 Definicija tablice činjenica (Fact Table)

In [3]:
class FactSupportTickets(Base):
    """
    Tablica činjenica — jedan redak = jedan helpdesk ticket.
    
    Grain: jedan ticket.
    
    FK ključevi:
        - reporter_key → dim_tehnicar (tko je prijavio)
        - assignee_key → dim_tehnicar (tko rješava, nullable — 46% ticketa nema assigneea)
        - projekt_prioritet_status_key → dim_projekt_prioritet_status (junk dimenzija)
        - vrijeme_key → dim_vrijeme
    
    Metrike:
        - vrijeme_rjesavanja_sati — ukupno vrijeme od kreiranja do rezolucije (sati)
        - broj_komentara — broj komentara na ticketu
        - sati_open — vrijeme provedeno u stanju 'open' (sati)
        - sati_in_progress — vrijeme u stanju 'in_progress' (sati)
        - sati_resolved — vrijeme u stanju 'resolved' (sati)
        - sati_waiting — vrijeme u stanju 'waiting' (sati)
    """
    __tablename__ = 'fact_support_tickets'

    ticket_id = Column(Integer, primary_key=True)

    # Strani ključevi prema dimenzijama
    reporter_key = Column(Integer, ForeignKey('dim_tehnicar.tehnicar_key'))
    assignee_key = Column(Integer, ForeignKey('dim_tehnicar.tehnicar_key'))
    projekt_prioritet_status_key = Column(Integer, ForeignKey('dim_projekt_prioritet_status.projekt_prioritet_status_key'))
    tip_ticketa_key = Column(Integer, ForeignKey('dim_tip_ticketa.tip_ticketa_key'))
    rezolucija_key = Column(Integer, ForeignKey('dim_rezolucija.rezolucija_key'))
    vrijeme_key = Column(Date, ForeignKey('dim_vrijeme.vrijeme_key'))

    # Metrike
    vrijeme_rjesavanja_sati = Column(Float)
    broj_komentara = Column(Integer)
    sati_open = Column(Float)
    sati_in_progress = Column(Float)
    sati_resolved = Column(Float)
    sati_waiting = Column(Float)


print("Fact klasa definirana: FactSupportTickets")

Fact klasa definirana: FactSupportTickets


## 5.3 Kreiranje tablica u bazi

In [4]:
from sqlalchemy import text

# Prvo dropaj stare tablice (sve moguće varijante iz prijašnjih verzija)
drop_statements = [
    "DROP TABLE IF EXISTS fact_support_tickets",
    "DROP TABLE IF EXISTS dim_vrijeme",
    "DROP TABLE IF EXISTS dim_projekt",
    "DROP TABLE IF EXISTS dim_tehnicar",
    "DROP TABLE IF EXISTS dim_prioritet_status",
    "DROP TABLE IF EXISTS dim_prioritet",
    "DROP TABLE IF EXISTS dim_status",
    "DROP TABLE IF EXISTS dim_projekt_prioritet_status",
    "DROP TABLE IF EXISTS dim_tip_ticketa",
    "DROP TABLE IF EXISTS dim_rezolucija",
]

with engine.connect() as conn:
    for stmt in drop_statements:
        conn.execute(text(stmt))
    conn.commit()
    print("Stare tablice obrisane.")

# Kreiraj nove tablice iz ORM definicija
Base.metadata.create_all(engine)
print("Sve tablice dimenzijskog modela su uspješno kreirane!")

Stare tablice obrisane.
Sve tablice dimenzijskog modela su uspješno kreirane!


## 5.4 Verifikacija kreiranih tablica

In [5]:
from sqlalchemy import inspect

inspector = inspect(engine)
tables = inspector.get_table_names()

print("Kreirane tablice:")
for tbl in tables:
    if tbl.startswith('dim_') or tbl.startswith('fact_'):
        cols = inspector.get_columns(tbl)
        col_names = [c['name'] for c in cols]
        fks = inspector.get_foreign_keys(tbl)
        print(f"  {tbl:40s} {len(cols)} stupaca, {len(fks)} FK  →  {col_names}")

Kreirane tablice:
  dim_projekt_prioritet_status             4 stupaca, 0 FK  →  ['projekt_prioritet_status_key', 'naziv_projekta', 'razina_prioriteta', 'naziv_statusa']
  dim_rezolucija                           3 stupaca, 0 FK  →  ['rezolucija_key', 'naziv_rezolucije', 'naziv_statusa']
  dim_tehnicar                             2 stupaca, 0 FK  →  ['tehnicar_key', 'ime_prezime']
  dim_tip_ticketa                          2 stupaca, 0 FK  →  ['tip_ticketa_key', 'naziv_tipa']
  dim_vrijeme                              6 stupaca, 0 FK  →  ['vrijeme_key', 'dan', 'mjesec', 'godina', 'kvartal', 'dan_u_tjednu']
  fact_support_tickets                     13 stupaca, 6 FK  →  ['ticket_id', 'reporter_key', 'assignee_key', 'projekt_prioritet_status_key', 'tip_ticketa_key', 'rezolucija_key', 'vrijeme_key', 'vrijeme_rjesavanja_sati', 'broj_komentara', 'sati_open', 'sati_in_progress', 'sati_resolved', 'sati_waiting']
